# Avelo — Data Engineer Screening Exercise

**Time:** ~60 minutes &nbsp;|&nbsp; **Format:** work entirely in this notebook — code *and* short written answers.

## How this works

- **Sections 1–4 are the exercise.** The **Stretch** section at the end is genuinely optional — it is not required for a strong result, and we would rather see Sections 1–4 done well than all five done thinly.
- **Show your work.** We care more about how you reason than about the final number. Leave your scratch cells, comments, and wrong turns in place — deleting them removes the most interesting evidence.
- **If you run out of time, stop and say so.** Add a note in the last cell describing what you would have done next. That note is scored. An honest "here is where I got stuck and here is how I would attack it" beats a silent blank.
- **You may use documentation and AI assistants**, exactly as you would on the job. We will walk through your answers together on a follow-up call and ask *why* — so make sure you can defend anything you submit, including anything a tool wrote for you. Code you cannot explain will not count in your favor.

## What we are looking at

| Section | Focus | Time |
|---|---|---|
| 1 | Reading unfamiliar code and reconciling a wrong number | ~20 min |
| 2 | Python: cleaning messy input | ~10 min |
| 3 | SQL: aggregation and window functions | ~20 min |
| 4 | Judgment: how you would handle a recurring load | ~10 min |
| Stretch | Optional. Attempt only if you have time left. | — |

*All data here is synthetic. No real passenger, crew, or booking data is used.*

---
## Setup — run this first

Loads four small CSVs from `data/`. No external connection and no credentials required.
If `import duckdb` fails, run `uv sync` in this folder (or `pip install duckdb pandas`).

In [9]:
import json
from pathlib import Path

import duckdb
import pandas as pd

DATA = Path("data")

flights = pd.read_csv(DATA / "flights.csv", parse_dates=["flight_date"])
bookings = pd.read_csv(DATA / "bookings.csv", parse_dates=["booked_at"])
raw = pd.read_csv(DATA / "raw_bookings.csv")
finance_control = pd.read_csv(DATA / "finance_control.csv")
segments = [tuple(s) for s in json.loads((DATA / "segments.json").read_text())]

con = duckdb.connect()
con.register("flights", flights)
con.register("bookings", bookings)


def q(sql):
    """Run SQL against the flights / bookings tables and return a DataFrame."""
    return con.execute(sql).df()


print("flights:", flights.shape, "| bookings:", bookings.shape)
flights.head(3)

flights: (366, 8) | bookings: (48369, 5)


,flight_id,flight_date,origin,dest,distance_mi,capacity,seats_sold,status
0,1000,2026-04-01,BUR,LAS,236,147,109,scheduled
1,1001,2026-04-02,BUR,LAS,236,147,105,scheduled
2,1002,2026-04-03,BUR,LAS,236,147,128,scheduled


**Schema**

- `flights(flight_id, flight_date, origin, dest, distance_mi, capacity, seats_sold, status)` — one row per scheduled flight, April–May 2026. `status` is `scheduled` or `cancelled`; a cancelled flight keeps the seats it had sold before it was cancelled.
- `bookings(booking_id, flight_id, fare_usd, booked_at, channel)` — one row per seat sold. `booked_at` is when the ticket was purchased; `flight_date` is when the aircraft departs.

Both are registered as SQL tables and available as pandas DataFrames — use whichever you prefer.

---
## Section 1 — The number is wrong  *(~20 min)*

A colleague wrote `build_route_summary()` below to produce April 2026 figures by route. It runs
without error and the output looks reasonable.

Finance publishes its own April figures independently, and **they do not match**. Finance's numbers
are correct; they are loaded as `finance_control`.

### Your task

1. Find out why the two disagree. **There is more than one cause.**
2. Fix the function so its output matches `finance_control` on every route, for every column.
3. In the markdown cell after your fix, write one short bullet per bug: **what was wrong, how you
   found it, and what it would have cost** if it had shipped to a dashboard unnoticed.

Step 3 matters as much as the fix. How you localized each problem is the part we will ask about.

In [10]:
def build_route_summary(flights, bookings):
    """April 2026 seats, capacity, load factor and revenue by route."""
    df = flights.merge(bookings, on="flight_id", how="inner")
    # bug: wrong date column (Finance used "flight_date" but here we use "booked_at", honestly both are valid) is used
    # bug: Finance only counted the status = "scheduled" flights
    df = df[(df["booked_at"] >= "2026-04-01") & (df["booked_at"] <= "2026-04-30")]
    df["route"] = df["origin"] + "-" + df["dest"]

    summary = df.groupby("route", as_index=False).agg(
        # bug: summing flight-level columns after joining to booking-level rows, "seat_sold", "capacity" are so off because of this
        seats_sold=("seats_sold", "sum"), 
        capacity=("capacity", "sum"),
        revenue_usd=("fare_usd", "sum"),
    )
    summary["load_factor_pct"] = round(
        100 * summary["seats_sold"] / summary["capacity"], 1
    )
    summary["revenue_usd"] = round(summary["revenue_usd"], 2)
    return summary.sort_values("load_factor_pct", ascending=False).reset_index(drop=True)


build_route_summary(flights, bookings)

,route,seats_sold,capacity,revenue_usd,load_factor_pct
0,HVN-MCO,893347,956718,724118.86,93.4
1,MCO-HVN,843125,932526,703918.29,90.4
2,BUR-LAS,360303,476721,229051.03,75.6
3,LAS-BUR,341448,456141,217808.11,74.9
4,HVN-TPA,499128,707805,536168.29,70.5
5,RDU-MCO,259039,395283,189877.78,65.5


In [ ]:

# code in SQL:
# SELECT
#     f.origin || '-' || f.dest AS route,
#     CAST(SUM(f.seats_sold) AS BIGINT) AS seats_sold,
#     CAST(SUM(f.capacity) AS BIGINT) AS capacity,
#     ROUND(SUM(b.fare_usd), 2) AS revenue_usd,
#     ROUND(100.0 * SUM(f.seats_sold) / SUM(f.capacity), 1) AS load_factor_pct
# FROM flights f
# INNER JOIN bookings b ON f.flight_id = b.flight_id
# WHERE b.booked_at >= '2026-04-01'
#   AND b.booked_at <= '2026-04-30'
# GROUP BY f.origin, f.dest
# ORDER BY load_factor_pct DESC


"""April 2026 seats, capacity, load factor and revenue by route."""
df = flights.merge(bookings, on="flight_id", how="inner")
df = df[(df["booked_at"] >= "2026-04-01") & (df["booked_at"] <= "2026-04-30")]
print(df.size)
df["route"] = df["origin"] + "-" + df["dest"]

# summary = df.groupby("route", as_index=False).agg(
#     seats_sold=("seats_sold", "sum"), 
#     capacity=("capacity", "sum"),
#     revenue_usd=("fare_usd", "sum"),
# )
# summary["load_factor_pct"] = round(
#     100 * summary["seats_sold"] / summary["capacity"], 1
# )
# summary["revenue_usd"] = round(summary["revenue_usd"], 2)

    # return summary.sort_values("load_factor_pct", ascending=False).reset_index(drop=True)

q("""
SELECT
    f.origin || '-' || f.dest AS route,
    CAST(SUM(f.seats_sold) AS BIGINT) AS seats_sold,
    CAST(SUM(f.capacity) AS BIGINT) AS capacity,
    ROUND(SUM(b.fare_usd), 2) AS revenue_usd,
    ROUND(100.0 * SUM(f.seats_sold) / SUM(f.capacity), 1) AS load_factor_pct
FROM flights f INNER JOIN bookings b ON f.flight_id = b.flight_id
WHERE 
    b.booked_at >= '2026-04-01'
  AND 
    b.booked_at <= '2026-04-30'
GROUP BY f.origin, f.dest
ORDER BY load_factor_pct DESC
""")




273312


,route,seats_sold,capacity,revenue_usd,load_factor_pct
0,HVN-MCO,893347,956718,724118.86,93.4
1,MCO-HVN,843125,932526,703918.29,90.4
2,BUR-LAS,360303,476721,229051.03,75.6
3,LAS-BUR,341448,456141,217808.11,74.9
4,HVN-TPA,499128,707805,536168.29,70.5
5,RDU-MCO,259039,395283,189877.78,65.5


In [12]:
finance_control   # the correct April 2026 figures — reconcile to this

,route,seats_sold,capacity,load_factor_pct,revenue_usd
0,HVN-MCO,5235,5670,92.3,741101.55
1,MCO-HVN,5093,5670,89.8,717098.08
2,BUR-LAS,3308,4410,75.0,230317.62
3,LAS-BUR,3272,4410,74.2,226817.64
4,HVN-TPA,3824,5481,69.8,536565.62
5,RDU-MCO,2671,4116,64.9,185574.29


In [19]:
# your investigation and your fixed version here

# My investigation:
# I noticed "seat_sold" and "capacity" is off by a lot, I ranslated the code to SQL, 
# then pin down seats_sold=("seats_sold", "sum"),  capacity=("capacity", "sum") is the cause
# For revenue_usd, it is less off, it could be a simply filter issue, then I pin down the "status" and wrong date column might be the issue
# I am more comfortable with SQL, so I create the SQL version first then translate it into Pandas


# def build_route_summary_fix(flights, bookings):
#     """April 2026 seats, capacity, load factor and revenue by route."""
#     fl = flights[
#         (flights["flight_date"] >= "2026-04-01")
#         & (flights["flight_date"] < "2026-05-01")
#         & (flights["status"] == "scheduled")
#     ].copy()
#     fl["route"] = fl["origin"] + "-" + fl["dest"]

   
#     rev = bookings.groupby("flight_id", as_index=False).agg(revenue_usd=("fare_usd", "sum"))
#     fl = fl.merge(rev, on="flight_id", how="left", validate="one_to_one")

#     summary = fl.groupby("route", as_index=False).agg(
#         seats_sold=("seats_sold", "sum"),
#         capacity=("capacity", "sum"),
#         revenue_usd=("revenue_usd", "sum"),
#     )
#     summary["load_factor_pct"] = round(
#         100 * summary["seats_sold"] / summary["capacity"], 1
#     )
#     summary["revenue_usd"] = round(summary["revenue_usd"], 2)
#     return summary.sort_values("load_factor_pct", ascending=False).reset_index(drop=True)

# build_route_summary_fix(flights,bookings)

q("""
WITH flight_revenue AS (
    SELECT flight_id, SUM(fare_usd) AS revenue
    FROM bookings
    GROUP BY flight_id
)
SELECT 

    f.origin || '-' || f.dest AS route,
    CAST(SUM(seats_sold) AS BIGINT) as seats_sold, 
    CAST(SUM(capacity) AS BIGINT) as capacity,
    ROUND(SUM(f.seats_sold)/SUM(f.capacity) * 100, 1) as load_factor_pct,
    ROUND(SUM(r.revenue), 2) as revenue

FROM flights f INNER JOIN flight_revenue r on f.flight_id = r.flight_id

WHERE 
    f.status == 'scheduled'
AND
    f.flight_date BETWEEN '2026-04-01' AND '2026-04-30'

GROUP BY 
    f.origin, f.dest, f.capacity

ORDER BY 
    revenue DESC
""")


,route,seats_sold,capacity,load_factor_pct,revenue
0,HVN-MCO,5235,5670,92.3,741101.55
1,MCO-HVN,5093,5670,89.8,717098.08
2,HVN-TPA,3824,5481,69.8,536565.62
3,BUR-LAS,3308,4410,75.0,230317.62
4,LAS-BUR,3272,4410,74.2,226817.64
5,RDU-MCO,2671,4116,64.9,185574.29


**What was wrong (one bullet per bug — what, how you found it, what it would have cost):**
#### bugs: ####
- wrong date column (Finance used "flight_date" but here we use "booked_at", honestly both are valid) is used
- Finance only counted the status = "scheduled" flights
- summing flight-level columns after joining to booking-level rows, "seat_sold", "capacity" are so off because of this
#### how I found (and fixed) it: ####
- I drew a relational diagram, translate the code to SQL (I am more familiar with that) to help me pin down possible errors
#### What it will cost: ####
- It will shot the charts in dashboard to sky high. If I am a upper management about to head to a meeting this will not look good.

---
## Section 2 — Clean the raw bookings  *(~10 min)*

`raw` is a small dump of booking records with the usual real-world defects. Produce a cleaned
DataFrame `clean` that a downstream pipeline could trust. At minimum:

- Normalize `origin` / `dest` to clean 3-letter uppercase codes.
- Coerce `fare_usd` to a numeric type.
- Handle any duplicate records.
- Normalize `channel` to a small consistent set.
- Make a defensible call on the **nulls** and on the **`$0` fare** — there is no single right answer,
  so **state your rule in a comment**. A rule we disagree with but that you argued for scores better
  than a silent drop.

In [14]:
raw   # preview

,booking_id,origin,dest,fare_usd,channel
0,1,bur,LAS,$89.00,app
1,2,HVN,mco,129,WEB
2,2,HVN,mco,129,WEB
3,3,hvn,TPA,NaN,app
4,4,MCO,hvn,$45.5,App
5,5,LAS,bur,72.00,web
6,6,rdu,MCO,$0,app
7,7,BUR,LAS,95,NaN
8,8,HVN,TPA,NaN,web
9,9,las,BUR,$61.25,APP


In [15]:
# your answer here

def fix_raw(raw_df):
    # Normalize `origin` / `dest` to clean 3-letter uppercase codes.
    for col in ["origin", "dest"]:
        raw_df[col] = raw_df[col].str.strip().str.upper()

    # Coerce `fare_usd` to a numeric type.

    # if fare_usd == NULL, flag it, push it to quarantine table, and investigate futher before injecting back to silver layer tables
    # if fare_usd == 0, it could happene if say any coupons or mileage redeemed adds up. I will consider this is valid.
    # Handle any duplicate records.
    raw_df["fare_usd"] = pd.to_numeric(
        raw_df["fare_usd"].str.strip().str.replace("$", "", regex=False)
    )
    
  
    raw_df = raw_df.drop_duplicates().reset_index(drop=True)
    
    # Normalize `channel` to a small consistent set. -> "web", "app", "other"
    raw_df["channel"] = raw_df["channel"].str.strip().str.lower().fillna("other")

    # I will add an alert here to check if there's unexpected value occur for each of
    # the columns (ex. Null values in booking_id)
    
    return raw_df

raw_df = pd.read_csv(DATA / "raw_bookings.csv")
fix_raw(raw_df)



,booking_id,origin,dest,fare_usd,channel
0,1,BUR,LAS,89.00,app
1,2,HVN,MCO,129.00,web
2,3,HVN,TPA,NaN,app
3,4,MCO,HVN,45.50,app
4,5,LAS,BUR,72.00,web
5,6,RDU,MCO,0.00,app
6,7,BUR,LAS,95.00,other
7,8,HVN,TPA,NaN,web
8,9,LAS,BUR,61.25,app
9,10,MCO,HVN,140.00,web


---
## Section 3 — SQL  *(~20 min)*

Use the `q()` helper. It runs **DuckDB**, which supports standard analytical SQL including window
functions and `QUALIFY`. Write ordinary ANSI-style SQL; no T-SQL-specific syntax needed.

If you would rather answer in pandas than SQL, say so in a comment and do that instead — we would
rather see a correct answer in your stronger language than a broken one in your weaker.

### 3a. Load factor by route (April 2026)

For **April 2026 only**, return one row per route (`origin-dest`) with total `seats_sold`, total
`capacity`, and **load factor %** (seats sold ÷ capacity), sorted by load factor descending.

In [16]:
q("""

SELECT 

    f.origin || '-' || f.dest AS route,
    CAST(SUM(seats_sold) AS BIGINT) as seats_sold, 
    CAST(SUM(capacity) AS BIGINT) as capacity,
    ROUND(SUM(f.seats_sold)/SUM(f.capacity) * 100, 1) as load_factor_pct

FROM flights f

WHERE 
    f.flight_date >= '2026-04-01' and f.flight_date < '2026-05-01' and f.status == 'scheduled'

GROUP BY 
    f.origin, f.dest, f.capacity

ORDER BY 
    load_factor_pct DESC

""")

,route,seats_sold,capacity,load_factor_pct
0,HVN-MCO,5235,5670,92.3
1,MCO-HVN,5093,5670,89.8
2,BUR-LAS,3308,4410,75.0
3,LAS-BUR,3272,4410,74.2
4,HVN-TPA,3824,5481,69.8
5,RDU-MCO,2671,4116,64.9


### 3b. Top revenue flights per route

Return the **two highest-revenue flights on each route across the full date range**, where a flight's revenue is the sum of its
booking fares. Include `route`, `flight_date`, and `revenue`.

Then answer in a comment: **if two flights on the same route tie exactly for second place, what does
your query return, and is that the behavior you want?**

In [17]:
q("""

WITH flight_revenue AS (
    SELECT flight_id, SUM(fare_usd) AS revenue
    FROM bookings
    GROUP BY flight_id
)
SELECT 

    f.origin || '-' || f.dest AS route
  
FROM flights f 
INNER JOIN flight_revenue r on f.flight_id = r.flight_id

WHERE f.status == 'scheduled' # QUALIFY RANK() OVER (PARTITION BY f.origin, f.dest ORDER BY r.revenue DESC) <= 2

ORDER BY 
    route, revenue DESC

LIMIT 2

""")
## 11/4:
## if two flights on the same route tie exactly for second place, what does your query return, and is that the behavior you want?
## A tie is possible: a route has many flights (flight_ids), and revenue is a calculated sum. 
## My query will simply return 2 rows in total (LIMIT 2) with one of the tie route (since 2 rows look exactly the same), but they might have different flight_id
## In terms of behavior, I will add other tie breakers like order by flight_date

## 11/5:
## I read the question wrong, I am supposed to get 12 rows instead of 2
## because the question is asking for top 2 revenues each route
## my query is close, I just need to do this for each route, so the right fix will be
## using QUALIFY, RANK(), OVER, PARTITION BY 

ParserException: Parser Error: syntax error at or near "#"

LINE 15: WHERE f.status == 'scheduled' # QUALIFY RANK() OVER (PARTITION BY f.origin, f.dest ORDER...
                                       ^

---
## Section 4 — Judgment  *(~10 min)*

**No prior Microsoft Fabric experience is expected or required.** We are hiring for reasoning, not
for tool trivia — answer from first principles. "I have not done this, but here is how I would think
about it" is a complete answer.

**Scenario.** Every morning at 06:00 a vendor drops a flat file of yesterday's bookings into cloud
storage. A job picks it up and loads it into the `bookings` table behind an executive dashboard.

Answer in the markdown cell below — bullets are fine, a few sentences each.

1. **The file arrives twice.** One morning the vendor sends the same file at 06:00 and again at
   06:20. What happens to the dashboard, and how would you design the load so that it does not
   matter how many times the same file is processed?
2. **The file is late.** It shows up at 11:00 instead of 06:00. What should the job — and the
   dashboard — do between 06:00 and 11:00? What is the worst outcome here, and is it worse than
   the job simply failing?
3. **The file gains a column.** Without warning, the vendor adds `promo_code`. Should the load
   accept it automatically, reject the file, or something else? Justify the choice.
4. **Someone has to trust this.** An executive asks you at 09:00 whether the number on the
   dashboard is right. What would you have put in place beforehand so that you can answer in
   under a minute, without opening a notebook?

_Your answers:_

**1. Arrives twice**

- Without protection, every booking loads twice, so the dashboard shows double bookings and double revenue, and nothing errors, so nobody notices. I’d protect it at two levels. First, a load log: when a file lands, I record its name and a checksum of its contents. If that exact file has already been loaded, the job skips it.

- Second, the load itself is an upsert keyed on booking ID, run in one transaction. So even if the same bookings come through again, they overwrite the existing rows instead of adding new ones, and a failed run rolls back cleanly. That way, processing the file once or ten times gives the same result.

**2. Arrives late**
- Worst outcome: the dashboard quietly shows yesterday’s numbers as if they were today’s, and an executive walks into a 9 a.m. meeting and decides on stale data without knowing it. That’s worse than the job failing, because a failure is visible and stale data looks normal.
- What I’d build: if the file hasn’t arrived by about 7:00, alert the data team. The dashboard shows “Data as of [yesterday], today’s file is late” next to the last-updated time, so the problem is visible even if nobody reads the alert. When the file lands at 11:00, an event trigger loads it right away, and the warning clears.

**3. Extra column**
- It really depends. Either options have pros and cons. Ideally, accept it, but don’t add it automatically. 
- An added column like promo_code doesn’t break anything we already use. Rejecting the file would leave executives with no bookings that day, So I’d load the columns we know, keep the full raw file with promo_code in bronze so nothing is lost, and alert the team.

**4. Trusting the number**
- I will create a data health log in the dashboard, something like "last updated at 11/16/2026, with new 100 bookings from yesterday"

---
# Stretch — optional

**Only attempt these if you have finished Sections 1–4 and have time left.** Leaving this entire
section blank does not count against you.

### S1. Sold-out streaks

A flight is **sold out** when `seats_sold >= capacity`. For each route, find every run of
**consecutive calendar days** that were sold out, returning `route`, `streak_start`, `streak_end`,
and `streak_length` — only for streaks of **2 or more days**, longest first.

In [ ]:
q("""
-- your SQL here
""")

### S2. Reconstruct the itinerary

`segments` is an unordered list of one traveler's flight legs as `(origin, dest)` pairs. The legs
chain into a **single continuous path** — each airport is departed from at most once, and there are
no round trips.

Write a function returning the airports in travel order, start to finish. Then, in a comment: state
the **time complexity**, and describe what would break if the itinerary contained a round trip.

In [ ]:
print(segments)

# your answer here


[('MCO', 'HVN'), ('BUR', 'LAS'), ('LAS', 'MCO'), ('HVN', 'RDU')]


### S3. Scale

Suppose `bookings` were not 48,000 rows but 48 billion, spread across a cluster, and `flights` were
still small enough to fit in memory on one machine.

1. What changes about how you would compute Section 3a, and why? 

1. Which step in that computation is the expensive one on a cluster, and what makes it expensive?

No Spark experience required — answer from whatever distributed or large-data experience you have,
or from first principles.

_Your answer:_

1. Not really, because there's no join for 3a.

2. GROUP BY, due to a lot of shuffling

---
## Before you send this back

Use this cell for anything you want us to know: what you ran out of time for, what you would do
next, anything you were unsure about, or anything you think the exercise got wrong.

_Your notes:_

Overall, I am able to solved section 1-4, (and I tried my best on Strech) it actually felt very similar with what I used to do. 

What takes me the most time is translating between Pandas and SQL, since I am more familiar with is SQL. 

The other harder part for me is section 4, where I will have to think of how this will affect the other users.